# Autobot Gemma 4 SWE-Coder SOTA
## Autonomous Software Engineering Agent for the Gemma 4 Developer Agent Challenge
### Architecture: Dual-Agent Specialization (`code_analyzer` + `swe_coder`) with Enabled Gemma 4 Thinking


In [ ]:
import os, sys, shutil, zipfile, hashlib, json
from pathlib import Path

WORKING_DIR = Path('/kaggle/working').resolve()
AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'configs').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'prompts').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'sub_agents').mkdir(parents=True, exist_ok=True)

print(f'Initialized build directory at: {AGENT_DIR}')


In [ ]:
# 1. Write configs/sampling.yaml with enabled Gemma 4 chain-of-thought thinking
SAMPLING_YAML = 'temperature: 0.15\ntop_p: 0.92\ntop_k: 40\nmax_output_tokens: 16384\nseed: 42\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n'
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(SAMPLING_YAML.strip() + '\n', encoding='utf-8')

# 2. Write eval_config.yaml with calibrated SWE budgets
EVAL_YAML = 'evaluation:\n  timeout_seconds: 180\n  max_tool_calls: 48\n  max_time_minutes: 8.0\n  max_turns: 60\n'
(AGENT_DIR / 'eval_config.yaml').write_text(EVAL_YAML.strip() + '\n', encoding='utf-8')

# 3. Write prompts/analyzer.md for root-cause localization
ANALYZER_MD = 'You are `code_analyzer`, a read-only code navigation and root-cause localization specialist in `/workspace`.\nYou never modify files or run tests. Your single purpose is to identify the exact file, line range, and defect mechanism for the reported issue.\n\n## Tools\n- `read_file(filepath, start_line, end_line)`: Inspect code with tight line windows (<= 150 lines).\n- `search_similar_code(query)`: Query semantic code embeddings using a single exact symbol name (e.g., `HTTPConnection`, `parse_header`), NOT natural language.\n- `get_code_neighbors(node)`: Inspect callers, callees, and imports around a known symbol.\n- `get_code_subgraph(nodes)`: View structural connections between candidate symbols.\n- `run_command(command)`: Use strictly for fast, read-only discovery commands (`rg -n "<symbol>"`, `git log -p -S <symbol>`, `find . -name "*.py"`).\n\n## Localization Protocol\n1. **Identifier Extraction**: Extract exact symbol names, class names, function names, error message fragments, and parameter names from the issue description and hints.\n2. **Targeted Search**:\n   - Use `rg -n "<exact_identifier>"` or `search_similar_code("<symbol_name>")`.\n   - Never search full natural language sentences.\n   - If the first symbol misses, search secondary symbols or error strings.\n3. **Trace Call Chain**: Follow callers/callees to locate where actual runtime behavior diverges from the specification.\n4. **Code Inspection**: Read the actual implementation with `read_file`. Confirm the exact line numbers and baseline logic.\n5. **Locate Target Test**: Identify existing test files under `tests/` that exercise this module.\n\n## Output Format\nConclude with a structured diagnosis (at most 250 words):\n- LOCATION: `<path>:<start_line>-<end_line>` (`<class_or_function>`)\n- ROOT CAUSE: `<concise 1-2 sentence explanation of the logical flaw>`\n- DEFECT TYPE: `<e.g. edge_case_handling | missing_validation | incorrect_type | off_by_one | API_drift>`\n- SUGGESTED FIX: `<concrete minimal change required in implementation source>`\n- TARGET TEST: `<path/to/test_file.py::test_function>`\n- CONFIDENCE: `HIGH | MEDIUM | LOW`\n'
(AGENT_DIR / 'prompts' / 'analyzer.md').write_text(ANALYZER_MD.strip() + '\n', encoding='utf-8')

# 4. Write prompts/system.md for root SWE-Coder
SYSTEM_MD = 'You are Autobot SWE-Coder, an elite autonomous software engineer in `/workspace`.\nYour goal is to solve the reported repository issue with a clean, minimal, working patch that passes hidden tests in a fresh verification environment.\nAlways conclude your work by calling `submit_patch()`.\n\n## Core Directives & Constraints\n\n1. **Strict Test Anti-Tampering Immunity**:\n   - The evaluation sandbox (`Container B`) forcefully wipes any changes made to test files (`test_*.py`, `*_test.py`, `tests/`), runner configurations (`pytest.ini`, `conftest.py`, `setup.cfg`, `pyproject.toml`), or package stubs.\n   - **NEVER modify or delete test files to make tests pass.**\n   - All code modifications MUST be strictly within the library/package implementation source code.\n\n2. **Workflow Phases**:\n   - **Phase 1: Scout & Diagnose**:\n     - Delegate initial localization to `code_analyzer` via your `agent_tool` or run focused `rg -n "<symbol>"` queries.\n     - Extract exact function signatures, error conditions, and expected behavior.\n     - Read the target source file with `read_file` to confirm line numbers and context.\n   - **Phase 2: Minimal Reproduction**:\n     - When feasible, run an inline behavior check: `python3 -c "..."` or run a targeted existing test: `pytest <target_test_file> -k "<test_function>" -v --tb=short`.\n     - Confirm the existing failure before applying modifications.\n     - Never create scratch scripts inside `/workspace`; use `/tmp` if temporary scripts are needed.\n   - **Phase 3: Surgical Implementation (`edit_file`)**:\n     - Modify the earliest incorrect layer in the source code.\n     - In `edit_file`, always include 3 to 5 lines of unchanged surrounding code before and after the modification in `old_string` to guarantee unique matching.\n     - Copy `old_string` character-for-character from `read_file`.\n     - Match surrounding code style, indentation, type annotations, and conventions.\n     - If an edit fails due to multiple matches, widen the surrounding context window.\n   - **Phase 4: Hermetic Verification**:\n     - First, verify Python syntax: `run_command("python3 -m py_compile <path/to/modified_file.py>")`.\n     - Second, run the targeted test: `run_command("pytest <target_test_file> -k \'<test_function>\' -v --tb=short")`.\n     - NEVER run the full repository test suite (this triggers timeouts). Run only the single targeted test.\n     - If the targeted test fails, analyze the traceback, refine the edit, and re-test.\n   - **Phase 5: Diff Audit & Clean Finish**:\n     - Inspect changes: `run_command("git status --short")` and `run_command("git diff --stat")`.\n     - Check for trailing whitespace or syntax warnings: `run_command("git diff --check")`.\n     - Remove any stray temporary files.\n     - **Call `submit_patch()`** as your final tool action. It captures all staged and unstaged diffs into the evaluation artifact.\n     - Provide a concise 2-sentence summary of the fix and verification outcome.\n\n3. **Tool & Budget Discipline**:\n   - Work swiftly: aim for 3–5 calls exploring, 2–4 calls editing, and 2–3 calls validating.\n   - The environment is completely offline. Pre-installed wheels exist under `/wheels`; never run `pip install` or internet commands.\n   - Single command timeout is 180 seconds.\n   - Call `get_status()` if approaching the budget limit to prioritize submitting the best defensible patch.\n'
(AGENT_DIR / 'prompts' / 'system.md').write_text(SYSTEM_MD.strip() + '\n', encoding='utf-8')

# 5. Write sub_agents/code_analyzer.yaml declarative agent
CODE_ANALYZER_YAML = 'name: code_analyzer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Read-only code navigation and root-cause localization specialist. Given an issue, returns exact file location, root cause, and target test.\ninstruction: !include ../prompts/analyzer.md\ngenerate_content_config: !include ../configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\n'
(AGENT_DIR / 'sub_agents' / 'code_analyzer.yaml').write_text(CODE_ANALYZER_YAML.strip() + '\n', encoding='utf-8')

# 6. Write root agent.yaml
AGENT_YAML = 'name: autobot_swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Autonomous software engineer that resolves repository issues with surgical, verified patches.\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: true\n'
(AGENT_DIR / 'agent.yaml').write_text(AGENT_YAML.strip() + '\n', encoding='utf-8')

print('Wrote all 6 declarative agent configuration and prompt files successfully.')


In [ ]:
# 7. Package submission.zip with deterministic timestamps
ZIP_PATH = WORKING_DIR / 'submission.zip'
ZIP_PATH.unlink(missing_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(p for p in AGENT_DIR.rglob('*') if p.is_file()):
        rel_path = str(p.relative_to(AGENT_DIR)).replace('\\', '/')
        info = zipfile.ZipInfo(rel_path, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        zf.writestr(info, p.read_bytes())

size_kb = ZIP_PATH.stat().st_size / 1024.0
print(f'Successfully created {ZIP_PATH.name} ({size_kb:.2f} KB)')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for name in zf.namelist():
        print(f'  - {name}')


In [ ]:
# 8. Rigorous validation against swegemma and adk-submission limits
import yaml

class SafeIncludeLoader(yaml.SafeLoader):
    pass
SafeIncludeLoader.add_constructor('!include', lambda loader, node: loader.construct_scalar(node))

# Verify root agent
with open(AGENT_DIR / 'agent.yaml', 'r', encoding='utf-8') as f:
    root_cfg = yaml.load(f, Loader=SafeIncludeLoader)
assert root_cfg['model'] == 'gemma-4-31b-it-qat-w4a16-ct', f"Invalid base model: {root_cfg['model']}"

# Verify code analyzer
with open(AGENT_DIR / 'sub_agents' / 'code_analyzer.yaml', 'r', encoding='utf-8') as f:
    sub_cfg = yaml.load(f, Loader=SafeIncludeLoader)
assert sub_cfg['model'] == 'gemma-4-31b-it-qat-w4a16-ct', f"Invalid sub-agent model: {sub_cfg['model']}"

# Verify submission size
MAX_SUBMISSION_BYTES = 3 * 1024 * 1024 * 1024
assert ZIP_PATH.stat().st_size < MAX_SUBMISSION_BYTES, 'Submission exceeds 3 GiB limit'

print('\n=========================================')
print('✅ ALL AUTOBOT SUBMISSION CHECKS PASSED!')
print(f'Artifact: {ZIP_PATH} is ready for scoring!')
print('=========================================')
